# Model Training & Inference

In [50]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import KFold, GridSearchCV
import warnings
warnings.filterwarnings('ignore')

## 1. Preprocessing


In [49]:
def preprocess(df: pd.DataFrame, fit_state: dict = None):
    is_fit = fit_state is None
    if is_fit:
        fit_state = {}

    df = df.copy()

    y = None
    if 'SalePrice' in df.columns:
        y = np.log1p(df.pop('SalePrice'))

    # ── Id ──────────────────────────────────────────────────────────────────
    df.drop(columns=['Id'], errors='ignore', inplace=True)

    # ── MasVnrType / MasVnrArea ─────────────────────────────────────────────
    df.loc[df['MasVnrType'].isna() & (df['MasVnrArea'] == 0),  'MasVnrType'] = 'None'
    df.loc[df['MasVnrType'].isna() & df['MasVnrArea'].isna(),  'MasVnrArea'] = 0
    df.loc[df['MasVnrType'].isna() & df['MasVnrArea'].isna(),  'MasVnrType'] = 'None'
    df.loc[df['MasVnrType'].isna() & (df['MasVnrArea'] > 0),   'MasVnrType'] = 'BrkFace'
    df.loc[(df['MasVnrType'] == 'None') & df['MasVnrArea'].isna(), 'MasVnrArea'] = 0

    # ── Basement NAs → 'None' where no basement ─────────────────────────────
    bsmt_cat_cols = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
    for col in bsmt_cat_cols:
        df.loc[(df['TotalBsmtSF'] == 0) & df[col].isna(), col] = 'None'

    # ── Garage NAs → 'None' where no garage ─────────────────────────────────
    garage_cat_cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
    for col in garage_cat_cols:
        df.loc[(df['GarageArea'] == 0) & df[col].isna(), col] = 'None'

    # ── Drop GarageYrBlt, PoolQC, PoolArea ──────────────────────────────────
    df.drop(['GarageYrBlt', 'PoolQC', 'PoolArea'], axis=1, inplace=True)

    # ── FireplaceQu: NA → 'None' where no fireplace ─────────────────────────
    df.loc[(df['Fireplaces'] == 0) & df['FireplaceQu'].isna(), 'FireplaceQu'] = 'None'

    # ── MiscFeature, Alley, Fence: NA → 'None' ──────────────────────────────
    for col in ['MiscFeature', 'Alley', 'Fence']:
        df.loc[df[col].isna(), col] = 'None'

    # ── LotFrontage: median imputation (fit on train) ────────────────────────
    if is_fit:
        fit_state['frontage_median'] = df['LotFrontage'].median()
    df['LotFrontage'].fillna(fit_state['frontage_median'], inplace=True)

    # ── Electrical: mode imputation (fit on train) ───────────────────────────
    if is_fit:
        fit_state['electrical_mode'] = df['Electrical'].mode()[0]
    df['Electrical'].fillna(fit_state['electrical_mode'], inplace=True)

    # ── Remaining basement NAs: mode from train ──────────────────────────────
    if is_fit:
        fit_state['bsmt_modes'] = {col: df[col].mode()[0] for col in bsmt_cat_cols}
    for col in bsmt_cat_cols:
        df[col].fillna(fit_state['bsmt_modes'][col], inplace=True)

    # ── Quality ordinal encoding ─────────────────────────────────────────────
    quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
    quality_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                    'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageCond', 'GarageQual']
    for col in quality_cols:
        df[col] = df[col].map(quality_map)

    # ── Other ordinal encodings ──────────────────────────────────────────────
    ordinal_maps = {
        'BsmtExposure': {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
        'BsmtFinType1': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
        'BsmtFinType2': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
        'GarageFinish': {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
        'LotShape':     {'IR3': 1, 'IR2': 2, 'IR1': 3, 'Reg': 4},
        'LandSlope':    {'Sev': 1, 'Mod': 2, 'Gtl': 3},
        'PavedDrive':   {'N': 1, 'P': 2, 'Y': 3},
        'Utilities':    {'ELO': 1, 'NoSeWa': 2, 'NoSewr': 3, 'AllPub': 4},
        'Functional':   {'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8},
        'Electrical':   {'Mix': 1, 'FuseP': 2, 'FuseF': 3, 'FuseA': 4, 'SBrkr': 5},
        'Fence':        {'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4},
    }
    for col, mapping in ordinal_maps.items():
        df[col] = df[col].map(mapping)

    # ── HasMultipleExterior, drop Exterior2nd ────────────────────────────────
    df['HasMultipleExterior'] = (df['Exterior1st'] != df['Exterior2nd']).astype(int)
    df.drop('Exterior2nd', axis=1, inplace=True)

    # ── HasMultipleCondition, drop Condition2 ────────────────────────────────
    df['HasMultipleCondition'] = (df['Condition1'] != df['Condition2']).astype(int)
    df.drop('Condition2', axis=1, inplace=True)

    # ── Binary encoding ──────────────────────────────────────────────────────
    df['Street']     = df['Street'].map({'Grvl': 0, 'Pave': 1})
    df['CentralAir'] = df['CentralAir'].map({'N': 0, 'Y': 1})

    # ── One-hot encode nominal features ──────────────────────────────────────
    nominal_cols = ['MSSubClass', 'MSZoning', 'Alley', 'LandContour', 'LotConfig',
                    'Neighborhood', 'Condition1', 'BldgType', 'HouseStyle',
                    'RoofStyle', 'RoofMatl', 'Exterior1st', 'MasVnrType',
                    'Foundation', 'Heating', 'GarageType', 'MiscFeature',
                    'SaleType', 'SaleCondition']
    df = pd.get_dummies(df, columns=nominal_cols, drop_first=True, dtype=int)

    # ── Align columns to training schema ─────────────────────────────────────
    if is_fit:
        fit_state['columns'] = df.columns.tolist()
    else:
        df = df.reindex(columns=fit_state['columns'], fill_value=0)


    df.fillna(0, inplace=True)

    return df, y, fit_state

    return df, y, fit_state

## 2. Load Data

In [51]:
train_raw = pd.read_csv('train.csv')
test_raw  = pd.read_csv('test.csv')

# Keep Id column separately for the submission file
test_ids = test_raw['Id'].copy()

print(f'Train shape: {train_raw.shape}')
print(f'Test  shape: {test_raw.shape}')

Train shape: (1460, 81)
Test  shape: (1459, 80)


## 3. Preprocess

In [52]:
# Fit on training data
X_train, y_train, fit_state = preprocess(train_raw, fit_state=None)

# Transform test data using statistics learned from train (no leakage)
X_test, _, fit_state = preprocess(test_raw, fit_state=fit_state)

print(f'X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'Remaining NAs — train: {X_train.isna().sum().sum()}  |  test: {X_test.isna().sum().sum()}')

X_train: (1460, 189)  |  y_train: (1460,)
X_test : (1459, 189)
Remaining NAs — train: 0  |  test: 0


In [54]:
X_test['Utilities'].fillna(X_test['Utilities'].mean(), inplace=True)

In [55]:
X_test.isna().mean()

LotFrontage              0.0
LotArea                  0.0
Street                   0.0
LotShape                 0.0
Utilities                0.0
                        ... 
SaleCondition_AdjLand    0.0
SaleCondition_Alloca     0.0
SaleCondition_Family     0.0
SaleCondition_Normal     0.0
SaleCondition_Partial    0.0
Length: 189, dtype: float64

## 4. Model Training

In [45]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Ridge()),
])

param_grid = [
    {
        'scaler':      [StandardScaler(), MinMaxScaler(), RobustScaler()],
        'model':       [Ridge()],
        'model__alpha': [0.5, 1, 2, 5, 10, 100],
    },
    {
        'scaler':      [StandardScaler(), MinMaxScaler(), RobustScaler()],
        'model':       [Lasso()],
        'model__alpha': [0.5, 1, 2, 5, 10, 100],
    },
]

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 36 candidates, totalling 180 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...l', Ridge())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'model': [Ridge()], 'model__alpha': [0.5, 1, ...], 'scaler': [StandardScaler(), MinMaxScaler(), ...]}, {'model': [Lasso()], 'model__alpha': [0.5, 1, ...], 'scaler': [StandardScaler(), MinMaxScaler(), ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuf

## 5. Results

In [56]:
best_params = grid_search.best_params_
best_cv_rmse = -grid_search.best_score_

print('Best parameters:')
for k, v in best_params.items():
    print(f'  {k}: {v}')
print(f'\nBest CV RMSE (log scale): {best_cv_rmse:.5f}')

Best parameters:
  model: Ridge()
  model__alpha: 5
  scaler: MinMaxScaler()

Best CV RMSE (log scale): 0.14386


## 6. Inference & Submission CSV

In [58]:
log_predictions = grid_search.predict(X_test)

predictions = np.expm1(log_predictions)

submission = pd.DataFrame({
    'Id':        test_ids.values,
    'SalePrice': predictions,
})

submission.to_csv('submission.csv', index=False)

submission.head(10)

submission1.csv saved!


,Id,SalePrice
0,1461,107451.181674
1,1462,140741.488516
2,1463,166199.362197
3,1464,189330.935334
4,1465,188667.065379
5,1466,162809.731114
6,1467,169402.516771
7,1468,159865.814247
8,1469,176204.572128
9,1470,121647.230490
